# Ex.1: Handwritten digit recognition with PyTorch
**EECE.4860/5860**: ungraded practice for Lecture 1. A small multilayer perceptron learns to classify MNIST digits.
This runs fine on a CPU (about 1 minute per epoch) or a GPU.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/acanets/genai-labs/blob/spring27/exercises/ex1_mnist.ipynb)

In [ ]:
import importlib.util
if importlib.util.find_spec("genai_utils") is None:
    get_ipython().run_line_magic("pip", 'install -q "git+https://github.com/acanets/genai-labs@spring27"')
import torch, torchvision
from torch import nn
import matplotlib.pyplot as plt
from genai_utils import setup_env, measure
device, _ = setup_env(need_gpu=False)

## 1. Data
Each image is 28 x 28 grayscale pixels; we scale them to [-1, 1].

In [ ]:
tf = torchvision.transforms.Compose([torchvision.transforms.ToTensor(), torchvision.transforms.Normalize((0.5,), (0.5,))])
train = torchvision.datasets.MNIST("data", train=True, download=True, transform=tf)
test = torchvision.datasets.MNIST("data", train=False, download=True, transform=tf)
train_dl = torch.utils.data.DataLoader(train, batch_size=64, shuffle=True)
test_dl = torch.utils.data.DataLoader(test, batch_size=1000)
x, y = next(iter(train_dl))
fig, ax = plt.subplots(1, 8, figsize=(8, 1.4))
for i in range(8):
    ax[i].imshow(x[i, 0], cmap="gray_r"); ax[i].set_title(int(y[i])); ax[i].axis("off")
plt.show()

## 2. Model, loss, optimizer
A 784-128-64-10 MLP with ReLU activations. `CrossEntropyLoss` combines log-softmax and negative log-likelihood. Try swapping `SGD` for `Adam` later and compare.

In [ ]:
model = nn.Sequential(nn.Flatten(), nn.Linear(784, 128), nn.ReLU(), nn.Linear(128, 64), nn.ReLU(), nn.Linear(64, 10)).to(device)
loss_fn = nn.CrossEntropyLoss()
opt = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9)
print(model, "\nparameters:", sum(p.numel() for p in model.parameters()))

## 3. Train: forward pass, loss, backward pass (gradients), optimizer step

In [ ]:
def accuracy():
    model.eval(); correct = 0
    with torch.no_grad():
        for xb, yb in test_dl:
            correct += (model(xb.to(device)).argmax(1) == yb.to(device)).sum().item()
    model.train(); return correct / len(test)

losses = []
with measure("training"):
    for epoch in range(3):
        for xb, yb in train_dl:
            xb, yb = xb.to(device), yb.to(device)
            loss = loss_fn(model(xb), yb)
            opt.zero_grad(); loss.backward(); opt.step()
            losses.append(loss.item())
        print(f"epoch {epoch + 1}: last loss {loss.item():.3f}, test accuracy {accuracy():.4f}")
plt.plot(losses); plt.xlabel("step"); plt.ylabel("training loss"); plt.show()

## 4. Try it
* Change the learning rate to 0.1 and 0.001. What happens to the loss curve?
* Replace SGD with `torch.optim.Adam(model.parameters(), lr=1e-3)`.
* Add a hidden layer. Does test accuracy improve?